# 02 — Training Loop

**Prerequisites**:
- `00_csi_pipeline.ipynb` — UNIFIED.jsonl + model definition
- `01_tokenization.ipynb` — token cache in `datasets/token_cache/`

### What this notebook does
1. **Load** token cache + config.yaml
2. **Init** GraphCodeBERTLoRACWEModel (LoRA, 0.24% trainable)
3. **Train** — AdamW + linear warmup, cross-entropy on 8-class CWE head
4. **Validate** — F1 / precision / recall per CWE class + macro-avg after each epoch
5. **Checkpoint** — save best model by macro F1
6. **Log** — metrics to JSON file (+ Google Drive if on Colab)

### Platform support
| Platform | Backend | Notes |
|---|---|---|
| Google Colab T4/A100 | **CUDA** | Recommended — full speed |
| Apple Silicon (M1/M2/M3) | **MPS** | ~3–5× slower than CUDA |
| CPU | **CPU** | Very slow — dev/debug only |

## 0 — Install Dependencies

In [ ]:
import subprocess, sys, platform

IS_APPLE_SILICON = platform.system() == "Darwin" and platform.machine() == "arm64"
IS_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in __import__("os").environ

print(f"Platform      : {platform.system()} {platform.machine()}")
print(f"Apple Silicon : {IS_APPLE_SILICON}")
print(f"Colab         : {IS_COLAB}")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers>=4.40",
        "peft>=0.10",
        "torch",
        "pyyaml",
        "scikit-learn",
        "tqdm",
    ],
    check=False,
)
print("Done.")

Platform      : Linux x86_64
Apple Silicon : False
Colab         : True
Done.


## 1 — Paths, Config & Device

In [ ]:
import os, json, platform, sys
from pathlib import Path
import yaml
import torch

IS_APPLE_SILICON = platform.system() == "Darwin" and platform.machine() == "arm64"

try:
    from google.colab import drive

    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/CSI_Project")
    IS_COLAB = True
    print("Colab — Drive mounted")
except ImportError:
    BASE_DIR = Path(os.path.dirname(os.path.abspath("__file__")))
    if not (BASE_DIR / "datasets").exists():
        BASE_DIR = Path.cwd()
    IS_COLAB = False
    print(f"Local — BASE_DIR: {BASE_DIR}")

# Load config
cfg_path = BASE_DIR / "config.yaml"
assert cfg_path.exists(), f"Missing config.yaml at {cfg_path}"
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

# Paths
TOKEN_CACHE_DIR = BASE_DIR / cfg["token_cache_dir"]
CHECKPOINT_DIR = BASE_DIR / cfg["checkpoint_dir"]
LOG_DIR = BASE_DIR / cfg["log_dir"]
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

MAX_LENGTH = cfg["max_length"]
CACHE_FILE = TOKEN_CACHE_DIR / f"tokens_maxlen{MAX_LENGTH}.pt"
assert (
    CACHE_FILE.exists()
), f"Missing cache: {CACHE_FILE} — run 01_tokenization.ipynb first"

# Device
if torch.cuda.is_available():
    DEVICE = "cuda"
elif IS_APPLE_SILICON and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

# T4 / CUDA performance flags
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True      # fastest conv algo for fixed input shape
    torch.set_float32_matmul_precision("high") # TF32 on Ampere; no-op on T4 but harmless
    print("cuDNN benchmark : ON")
    print("TF32 matmul     : high")

print(f"Device          : {DEVICE}")
print(f"CACHE_FILE      : {CACHE_FILE}")
print(f"CHECKPOINT_DIR  : {CHECKPOINT_DIR}")
print(f"LOG_DIR         : {LOG_DIR}")

# Reproducibility
import random, numpy as np

SEED = cfg["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)


Mounted at /content/drive
Colab — Drive mounted
cuDNN benchmark : ON
TF32 matmul     : high
Device          : cuda
CACHE_FILE      : /content/drive/MyDrive/CSI_Project/datasets/token_cache/tokens_maxlen512.pt
CHECKPOINT_DIR  : /content/drive/MyDrive/CSI_Project/checkpoints
LOG_DIR         : /content/drive/MyDrive/CSI_Project/logs


## 2 — Load Token Cache + Build DataLoaders

In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import numpy as np

class VulnerabilityDataset(Dataset):
    def __init__(self, cache: dict, split: str):
        if split == "all":
            indices = list(range(len(cache["split_origins"])))
        else:
            indices = [i for i, s in enumerate(cache["split_origins"]) if s == split]
        self.input_ids = cache["input_ids"][indices]
        self.attention_mask = cache["attention_mask"][indices]
        self.cwe_labels = cache["cwe_labels"][indices]
        self.binary_labels = cache["binary_labels"][indices]
        self.global_ids = cache["global_ids"][indices]
        self.split = split

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "cwe_label": self.cwe_labels[idx],
            "binary_label": self.binary_labels[idx],
            "global_id": self.global_ids[idx],
        }


def make_balanced_sampler(dataset):
    labels = dataset.cwe_labels.numpy()
    class_counts = np.bincount(labels, minlength=8)
    class_weights = 1.0 / (class_counts + 1e-6)
    sample_weights = class_weights[labels]
    return WeightedRandomSampler(
        weights=torch.tensor(sample_weights, dtype=torch.float),
        num_samples=len(labels),
        replacement=True,
    )


# Load cache
cache = torch.load(CACHE_FILE, weights_only=True)
print(f'Cache loaded: {cache["num_records"]:,} records')

# T4 has 16 GB VRAM — double batch size when using AMP (fp16 halves memory)
BATCH_SIZE = 32 if DEVICE == "cuda" else cfg["batch_size"]

train_ds = VulnerabilityDataset(cache, split="train")
val_ds = VulnerabilityDataset(cache, split="val")

# num_workers=2; no persistent_workers (Colab stability)
_loader_kw = dict(num_workers=2, pin_memory=(DEVICE == "cuda"),
                  persistent_workers=False, prefetch_factor=2)

sampler = make_balanced_sampler(train_ds)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, **_loader_kw)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,   **_loader_kw)

print(f"Batch size    : {BATCH_SIZE}")
print(f"Train batches : {len(train_loader):,}  ({len(train_ds):,} samples)")
print(f"Val batches   : {len(val_loader):,}  ({len(val_ds):,} samples)")

# Inverse-frequency class weights for weighted CrossEntropyLoss
_counts = np.bincount(train_ds.cwe_labels.numpy(), minlength=8).astype(float)
CLASS_WEIGHTS = torch.tensor(len(train_ds) / (8 * _counts), dtype=torch.float)
print(f"Class weights : {CLASS_WEIGHTS.numpy().round(3)}")


Cache loaded: 14,522 records
Batch size    : 32
Train batches : 407  (12,994 samples)
Val batches   : 48  (1,528 samples)
Class weights : [1.676 1.517 1.173 2.216 0.574 1.219 1.769 0.433]


## 3 — Model, Optimizer & Scheduler

In [ ]:
import torch.nn.functional as F
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from peft import LoraConfig, TaskType, get_peft_model
from typing import Dict

CWE_8_CLASSES = [
    "CWE-077",
    "CWE-601",
    "CWE-022",
    "CWE-094",
    "CWE-089",
    "CWE-352",
    "CWE-079",
    "unknown",
]
CWE_TO_INDEX = {cwe: i for i, cwe in enumerate(CWE_8_CLASSES)}
INDEX_TO_CWE = {i: cwe for cwe, i in CWE_TO_INDEX.items()}


class CWEClassificationHead(nn.Module):
    """Two-layer MLP: hidden → hidden/2 → num_classes, with LayerNorm + GELU."""
    def __init__(self, hidden_size: int, num_classes: int = 8, dropout: float = 0.1):
        super().__init__()
        mid = hidden_size // 2
        self.norm = nn.LayerNorm(hidden_size)
        self.fc1  = nn.Linear(hidden_size, mid)
        self.act  = nn.GELU()
        self.drop = nn.Dropout(dropout)
        self.fc2  = nn.Linear(mid, num_classes)

    def forward(self, x):
        x = self.norm(x)
        x = self.drop(self.act(self.fc1(x)))
        return self.fc2(x)


class GraphCodeBERTLoRACWEModel(nn.Module):
    def __init__(
        self,
        model_name: str = "microsoft/graphcodebert-base",
        num_cwe_classes: int = 8,
        lora_r: int = 16,
        lora_alpha: int = 32,
        lora_dropout: float = 0.1,
        class_weights=None,
        focal_gamma: float = 2.0,
    ):
        super().__init__()
        encoder = AutoModel.from_pretrained(model_name)
        lora_cfg = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            target_modules=["query", "key", "value"],  # added key
            bias="none",
        )
        self.encoder = get_peft_model(encoder, lora_cfg)
        hidden = self.encoder.config.hidden_size
        self.cwe_head   = CWEClassificationHead(hidden, num_cwe_classes)
        self.focal_gamma = focal_gamma
        self.register_buffer(
            "class_weights",
            class_weights if class_weights is not None else torch.ones(num_cwe_classes),
        )

    @staticmethod
    def _mean_pool(hidden_states, attention_mask):
        """Mean pool over non-padding tokens."""
        mask   = attention_mask.unsqueeze(-1).float()      # (B, L, 1)
        summed = (hidden_states * mask).sum(1)             # (B, H)
        counts = mask.sum(1).clamp(min=1e-9)               # (B, 1)
        return summed / counts                             # (B, H)

    def forward(self, input_ids, attention_mask, cwe_labels=None):
        enc       = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled    = self._mean_pool(enc.last_hidden_state, attention_mask)
        logits    = self.cwe_head(pooled)
        result    = {"logits": logits}
        if cwe_labels is not None:
            # Focal loss + class-frequency rebalancing
            ce_weighted   = F.cross_entropy(logits, cwe_labels,
                                            weight=self.class_weights, reduction="none")
            pt            = torch.exp(-F.cross_entropy(logits, cwe_labels, reduction="none"))
            result["loss"] = (((1 - pt) ** self.focal_gamma) * ce_weighted).mean()
        return result


def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable, total, 100 * trainable / total


# r=16, Q+K+V — starts fresh (checkpoint incompatible with new arch)
LORA_R     = 16
LORA_ALPHA = 32

model = GraphCodeBERTLoRACWEModel(
    model_name     = cfg["model_name"],
    num_cwe_classes= cfg["num_cwe_classes"],
    lora_r         = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = cfg["lora_dropout"],
    class_weights  = CLASS_WEIGHTS,
    focal_gamma    = 2.0,
).to(DEVICE)

trainable, total, pct = count_params(model)
print(f"Model loaded to    {DEVICE}")
print(f"Trainable params : {trainable:,} / {total:,}  ({pct:.2f}%)")
print(f"LoRA             : r={LORA_R}, alpha={LORA_ALPHA}, targets=Q+K+V")
print(f"Head             : two-layer MLP (LayerNorm→Linear→GELU→Dropout→Linear)")
print(f"Pooling          : mean pool over non-padding tokens")
print(f"Loss             : Focal (gamma=2.0) + class weights")

EPOCHS = 15
LR     = 2e-4  # higher LR for fresh LoRA weights; scheduler decays to 0

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * cfg["warmup_ratio"])

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=cfg["weight_decay"],
)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps  = warmup_steps,
    num_training_steps= total_steps,
)

print(f"\nEpochs : {EPOCHS}  |  LR : {LR}  |  Steps : {total_steps:,}  |  Warmup : {warmup_steps}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded to    cuda
Trainable params : 1,184,648 / 125,830,280  (0.94%)
LoRA             : r=16, alpha=32, targets=Q+K+V
Head             : two-layer MLP (LayerNorm→Linear→GELU→Dropout→Linear)
Pooling          : mean pool over non-padding tokens
Loss             : Focal (gamma=2.0) + class weights

Epochs : 15  |  LR : 0.0002  |  Steps : 6,105  |  Warmup : 610


## 4 — Train & Validate

In [ ]:
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report,
)
from tqdm.auto import tqdm
import time

MAX_GRAD_NORM = cfg["max_grad_norm"]
LOG_EVERY     = cfg["log_every_n_steps"]
SAVE_EVERY    = cfg["save_every_n_steps"]
PATIENCE      = cfg["early_stopping_patience"]

CHECKPOINT_DIR = BASE_DIR / cfg["checkpoint_dir"]
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
BEST_CKPT = CHECKPOINT_DIR / "best_model.pt"

USE_AMP = DEVICE == "cuda"
scaler  = torch.amp.GradScaler("cuda", enabled=USE_AMP) if USE_AMP else None
print(f" (fp16) : {'ON' if USE_AMP else 'OFF'}")

# ── Auto-resume (skip if architecture changed) ────────────────────────────────
BEST_F1     = 0.0
start_epoch = 1
global_step = 0
no_improve  = 0
history     = []

if BEST_CKPT.exists():
    print(f"Found checkpoint: {BEST_CKPT}")
    checkpoint = torch.load(BEST_CKPT, map_location=DEVICE)
    state_key  = "model_state_dict" if "model_state_dict" in checkpoint else "model_state"
    try:
        model.load_state_dict(checkpoint[state_key], strict=True)
        if "optimizer_state_dict" in checkpoint:
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        BEST_F1     = checkpoint["val_f1"]
        start_epoch = checkpoint["epoch"] + 1
        global_step = (start_epoch - 1) * len(train_loader)
        remaining   = (EPOCHS - start_epoch + 1) * len(train_loader)
        scheduler   = get_linear_schedule_with_warmup(
            optimizer, num_warmup_steps=0,
            num_training_steps=max(remaining, 1)
        )
        print(f"✓ Resumed epoch {start_epoch}, best F1={BEST_F1:.4f}")
    except RuntimeError:
        print("Architecture changed — starting fresh (checkpoint skipped)")
else:
    print("No checkpoint — starting fresh.")


def validate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    total_loss, n_batches = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            ids   = batch["input_ids"].to(device)
            mask  = batch["attention_mask"].to(device)
            lbls  = batch["cwe_label"].to(device)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                out = model(ids, mask, cwe_labels=lbls)
            total_loss += out["loss"].item()
            all_preds.extend(out["logits"].argmax(-1).cpu().tolist())
            all_labels.extend(lbls.cpu().tolist())
            n_batches += 1
    avg_loss   = total_loss / n_batches
    macro_f1   = f1_score(all_labels, all_preds, average="macro",    zero_division=0)
    macro_prec = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    macro_rec  = recall_score(all_labels, all_preds, average="macro",  zero_division=0)
    return avg_loss, macro_f1, macro_prec, macro_rec, all_preds, all_labels


for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    epoch_loss = step_loss = 0.0
    t0 = time.time()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=True)
    for step, batch in enumerate(pbar, 1):
        ids  = batch["input_ids"].to(DEVICE, non_blocking=True)
        mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        lbls = batch["cwe_label"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            out  = model(ids, mask, cwe_labels=lbls)
            loss = out["loss"]

        if USE_AMP:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()), MAX_GRAD_NORM
            )
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()), MAX_GRAD_NORM
            )
            optimizer.step()

        scheduler.step()
        epoch_loss += loss.item()
        step_loss  += loss.item()
        global_step += 1

        if step % LOG_EVERY == 0:
            pbar.set_postfix({"loss": f"{step_loss/LOG_EVERY:.4f}",
                              "lr":   f"{scheduler.get_last_lr()[0]:.2e}"})
            step_loss = 0.0

        if global_step % SAVE_EVERY == 0:
            ckpt = CHECKPOINT_DIR / f"step_{global_step}.pt"
            torch.save({"step": global_step, "epoch": epoch,
                        "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "scheduler_state_dict": scheduler.state_dict()}, ckpt)
            print(f"  Saved step checkpoint: {ckpt}")

    train_loss = epoch_loss / len(train_loader)
    val_loss, val_f1, val_prec, val_rec, preds, labels = validate(model, val_loader, DEVICE)
    elapsed = time.time() - t0

    print(
        f"\nEpoch {epoch:02d}  train={train_loss:.4f}  val={val_loss:.4f}  "
        f"F1={val_f1:.4f}  P={val_prec:.4f}  R={val_rec:.4f}  ({elapsed:.0f}s)"
    )
    print(classification_report(labels, preds, target_names=CWE_8_CLASSES, zero_division=0))

    history.append({"epoch": epoch, "train_loss": round(train_loss, 4),
                    "val_loss": round(val_loss, 4), "val_f1": round(val_f1, 4),
                    "val_prec": round(val_prec, 4), "val_rec": round(val_rec, 4),
                    "elapsed_s": round(elapsed, 1)})

    if val_f1 > BEST_F1:
        BEST_F1    = val_f1
        no_improve = 0
        torch.save({"epoch": epoch, "val_f1": val_f1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "config": cfg,
                    "arch": {"lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
                             "pooling": "mean", "head": "mlp2"}},
                   BEST_CKPT)
        print(f"  ✓ New best F1={val_f1:.4f} — saved")
    else:
        no_improve += 1
        print(f"  No improvement ({no_improve}/{PATIENCE})")
        if no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"\nDone. Best val F1 = {BEST_F1:.4f}")
print(f"History: {history}")


AMP (fp16) : ON
No checkpoint — starting fresh.


Epoch 1/15:   0%|          | 0/407 [00:00<?, ?it/s]

  Saved step checkpoint: /content/drive/MyDrive/CSI_Project/checkpoints/step_200.pt
  Saved step checkpoint: /content/drive/MyDrive/CSI_Project/checkpoints/step_400.pt

Epoch 01  train=1.3818  val=0.9145  F1=0.4907  P=0.5145  R=0.5360  (325s)
              precision    recall  f1-score   support

     CWE-077       0.45      0.54      0.49       153
     CWE-601       0.22      0.59      0.33        93
     CWE-022       0.43      0.50      0.46       103
     CWE-094       0.37      0.43      0.40       124
     CWE-089       0.89      0.58      0.70       350
     CWE-352       0.56      0.79      0.66       146
     CWE-079       0.32      0.51      0.40       152
     unknown       0.86      0.35      0.49       407

    accuracy                           0.51      1528
   macro avg       0.51      0.54      0.49      1528
weighted avg       0.64      0.51      0.53      1528

  ✓ New best F1=0.4907 — saved


Epoch 2/15:   0%|          | 0/407 [00:00<?, ?it/s]

  Saved step checkpoint: /content/drive/MyDrive/CSI_Project/checkpoints/step_600.pt
  Saved step checkpoint: /content/drive/MyDrive/CSI_Project/checkpoints/step_800.pt

Epoch 02  train=0.4708  val=1.0099  F1=0.5564  P=0.5627  R=0.5951  (341s)
              precision    recall  f1-score   support

     CWE-077       0.49      0.51      0.50       153
     CWE-601       0.35      0.61      0.45        93
     CWE-022       0.33      0.66      0.44       103
     CWE-094       0.35      0.52      0.42       124
     CWE-089       0.84      0.75      0.79       350
     CWE-352       0.73      0.82      0.77       146
     CWE-079       0.50      0.40      0.45       152
     unknown       0.91      0.49      0.64       407

    accuracy                           0.59      1528
   macro avg       0.56      0.60      0.56      1528
weighted avg       0.67      0.59      0.61      1528

  ✓ New best F1=0.5564 — saved


...

  Saved step checkpoint: /content/drive/MyDrive/CSI_Project/checkpoints/step_4200.pt
  Saved step checkpoint: /content/drive/MyDrive/CSI_Project/checkpoints/step_4400.pt

Epoch 11  train=0.0126  val=1.4853  F1=0.6923  P=0.7116  R=0.6975  (330s)
              precision    recall  f1-score   support

     CWE-077       0.72      0.59      0.65       153
     CWE-601       0.56      0.60      0.58        93
     CWE-022       0.53      0.72      0.61       103
     CWE-094       0.57      0.54      0.56       124
     CWE-089       0.81      0.85      0.83       350
     CWE-352       0.72      0.90      0.80       146
     CWE-079       0.89      0.47      0.61       152
     unknown       0.89      0.91      0.90       407

    accuracy                           0.76      1528
   macro avg       0.71      0.70      0.69      1528
weighted avg       0.77      0.76      0.75      1528

  ✓ New best F1=0.6923 — saved


Epoch 12/15:   0%|          | 0/407 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b2cecd0cea0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():Exception ignored in: 
 <function _MultiProcessingDataLoaderIter.__del__ at 0x7b2cecd0cea0>
  Traceback (most recent call last):
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
      self._shutdown_workers() 
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    ^if w.is_alive():^^
^  ^  ^ ^ ^ ^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^    ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^
^^ ^
   File "/usr/lib/pyth

  Saved step checkpoint: /content/drive/MyDrive/CSI_Project/checkpoints/step_4600.pt


KeyboardInterrupt: 

## 5 — Save Metrics Log

In [ ]:
import datetime

log = {
    "run_date": datetime.datetime.now().isoformat(),
    "device": DEVICE,
    "model_name": cfg["model_name"],
    "best_val_f1": BEST_F1,
    "config": cfg,
    "history": history,
}

log_path = LOG_DIR / f'run_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
with open(log_path, "w") as f:
    json.dump(log, f, indent=2)

print(f"Metrics saved: {log_path}")
print(f"Best val F1  : {BEST_F1:.4f}")
print(f"Best ckpt    : {BEST_CKPT}")

# Print history table
print(
    f'\n{"Epoch":>5} {"TrainLoss":>10} {"ValLoss":>10} {"ValF1":>8} {"ValP":>8} {"ValR":>8}'
)
for r in history:
    print(
        f'{r["epoch"]:>5} {r["train_loss"]:>10.4f} {r["val_loss"]:>10.4f} '
        f'{r["val_f1"]:>8.4f} {r["val_prec"]:>8.4f} {r["val_rec"]:>8.4f}'
    )

Metrics saved: /content/drive/MyDrive/CSI_Project/logs/run_20260420_171528.json
Best val F1  : 0.6923
Best ckpt    : /content/drive/MyDrive/CSI_Project/checkpoints/best_model.pt

Epoch  TrainLoss    ValLoss    ValF1     ValP     ValR
    1     1.3818     0.9145   0.4907   0.5145   0.5360
    2     0.4708     1.0099   0.5564   0.5627   0.5951
    3     0.2153     0.9475   0.6043   0.6019   0.6396
    4     0.1190     1.1849   0.6348   0.6402   0.6644
    5     0.0835     1.2089   0.6551   0.6589   0.6707
    6     0.0482     1.2011   0.6593   0.6617   0.6850
    7     0.0377     1.3554   0.6671   0.6712   0.6826
    8     0.0263     1.3470   0.6789   0.6905   0.6899
    9     0.0226     1.4356   0.6865   0.7005   0.6935
   10     0.0209     1.4324   0.6795   0.6948   0.6913
   11     0.0126     1.4853   0.6923   0.7116   0.6975


## 6 — Load Best Checkpoint (inference ready)

In [ ]:
# Load best checkpoint for inference / further evaluation
ckpt = torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)
state_key = "model_state_dict" if "model_state_dict" in ckpt else "model_state"
model.load_state_dict(ckpt[state_key])
model.eval()

print(f"Loaded best checkpoint")
print(f'  epoch   : {ckpt["epoch"]}')
print(f'  val_f1  : {ckpt["val_f1"]:.4f}')

# Quick inference example
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(cfg["model_name"])

sample_code = (
    "def get_user(uid): return db.execute('SELECT * FROM users WHERE id=' + uid)"
)
enc = tokenizer(
    sample_code,
    max_length=MAX_LENGTH,
    padding="max_length",
    truncation=True,
    return_tensors="pt",
)
enc = {k: v.to(DEVICE) for k, v in enc.items()}

with torch.no_grad():
    out = model(enc["input_ids"], enc["attention_mask"])
    pred = out["logits"].argmax(dim=-1).item()
    probs = torch.softmax(out["logits"], dim=-1)[0].tolist()

print(f"\nSample prediction:")
print(f"  code : {sample_code[:60]}...")
print(f"  pred : {INDEX_TO_CWE[pred]}  (index {pred})")
print(f"  top3 :")
top3 = sorted(enumerate(probs), key=lambda x: -x[1])[:3]
for idx, prob in top3:
    print(f"    {INDEX_TO_CWE[idx]:<12} {prob:.4f}")

Loaded best checkpoint
  epoch   : 11
  val_f1  : 0.6923


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]


Sample prediction:
  code : def get_user(uid): return db.execute('SELECT * FROM users WH...
  pred : CWE-089  (index 4)
  top3 :
    CWE-089      0.6413
    CWE-077      0.2207
    unknown      0.1358
